# IMDB / Netflix Shows Content-Based Recommender

In [1]:
import ast
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
df = pd.read_csv("imdb_processed.csv", low_memory=False)

In [3]:
df["genre"] = df["genre"].fillna("")
df["director"] = df["director"].fillna("")
df["cast"] = df["cast"].fillna("")

In [4]:
def create_soup(x):
    return (
        str(x["genre"])
        + " "
        + str(x["director"])
        + " "
        + str(x["cast"])
        + " "
        + str(x["title"])
    )

In [5]:
df["soup"] = df.apply(create_soup, axis=1)

In [6]:
tfidf = TfidfVectorizer(stop_words="english")
tfidf_matrix = tfidf.fit_transform(df["soup"])

In [7]:
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)

In [8]:
indices = pd.Series(df.index, index=df["title"]).drop_duplicates()

In [9]:
def get_recommendations(title, cosine_sim=cosine_sim):
    if title not in indices:
        return f"'{title}' veri setinde bulunamadı."

    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

    sim_scores = sim_scores[1:6]
    movie_indices = [i[0] for i in sim_scores]

    return df["title"].iloc[movie_indices]

In [10]:
sample_title = df["title"].iloc[0]
print(f"\nTarget Show: {sample_title}")
print("\nRecommended Shows:")
print(get_recommendations(sample_title))


Target Show: Dinosaur Planet

Recommended Shows:
2986                                       Living Planet
3417    3-2-1 Penguins: Moon Menace on Planet Tell-A-Lie
8378    3-2-1 Penguins! Trouble on Planet Wait-Your-Turn
5556                                              Horses
3961                                       Sacred Planet
Name: title, dtype: str
